# Kickstarter Success Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `success`

## 2 · Project Overview

We predict whether a **Kickstarter project will succeed** in reaching its funding goal based on campaign features: goal amount, duration, category, early backer count, video presence, and creator experience.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a Kickstarter campaign's goal, duration, category, early backers, video flag, description length, reward tiers, and creator history, predict whether it will be successfully funded.

## 5 · Why This Project Matters

- **Crowdfunding success prediction** helps creators optimize campaigns.
- Understanding success factors improves campaign design.
- Platform can recommend improvements to creators.
- Classic binary classification with mixed feature types.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | goal_usd, duration_days, category, backers_first_week, has_video, description_length, num_rewards, creator_prev_projects |
| **Target** | success (0 = failed, 1 = successful) |
| **Class balance** | ~50/50 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "success"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: success
Save dir: E:\Github\Machine-Learning-Projects\Classification\Kickstarter Success Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 Kickstarter projects with campaign features and success outcome.

In [4]:
np.random.seed(SEED)
n = 8000
goal_usd = np.round(np.random.lognormal(8, 1.2, n).clip(100, 500000), 0)
duration_days = np.random.randint(7, 60, n)
category = np.random.choice(["Technology", "Games", "Design", "Film", "Music", "Art", "Food"], n,
                             p=[0.2, 0.15, 0.15, 0.15, 0.1, 0.1, 0.15])
backers_first_week = np.random.poisson(15, n)
has_video = np.random.binomial(1, 0.6, n)
description_length = np.random.poisson(300, n).clip(50, 1000)
num_rewards = np.random.randint(1, 15, n)
creator_prev_projects = np.random.poisson(1, n)

cat_bonus = np.where(category == "Games", 0.3, np.where(category == "Technology", 0.2,
            np.where(category == "Design", 0.15, 0)))
score = (-0.000005 * goal_usd + 0.03 * backers_first_week + 0.2 * has_video
         + 0.001 * description_length + 0.05 * num_rewards
         + 0.15 * creator_prev_projects + cat_bonus - 0.005 * duration_days
         + np.random.normal(0, 0.8, n))
success = (score > np.median(score)).astype(int)

df = pd.DataFrame({"goal_usd": goal_usd, "duration_days": duration_days,
                    "category": category, "backers_first_week": backers_first_week,
                    "has_video": has_video, "description_length": description_length,
                    "num_rewards": num_rewards, "creator_prev_projects": creator_prev_projects,
                    "success": success})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['success'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (8000, 9)
Class balance:
success
1    0.5
0    0.5
Name: proportion, dtype: float64


,goal_usd,duration_days,category,backers_first_week,has_video,description_length,num_rewards,creator_prev_projects,success
0,5410.0,32,Film,20,0,275,12,2,1
1,2525.0,19,Film,13,1,329,8,3,0
2,6485.0,14,Design,25,1,302,4,1,1
3,18539.0,11,Design,21,1,308,10,0,0
4,2251.0,50,Technology,20,1,328,8,2,0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (8000, 9)

Missing values:
goal_usd                 0
duration_days            0
category                 0
backers_first_week       0
has_video                0
description_length       0
num_rewards              0
creator_prev_projects    0
success                  0
dtype: int64

Duplicate rows: 0

Dtypes:
goal_usd                 float64
duration_days              int32
category                  object
backers_first_week         int32
has_video                  int32
description_length         int32
num_rewards                int32
creator_prev_projects      int32
success                    int64
dtype: object

Target 'success' confirmed. Value counts:
success
1    4000
0    4000
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["goal_usd", "duration_days", "backers_first_week", "description_length", "num_rewards"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
axes[1][2].axis("off")
plt.suptitle("Feature Distributions by Success Status", fontsize=14)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
df.groupby("category")[TARGET].mean().sort_values().plot(kind="barh", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Success Rate by Category")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `success`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["salmon", "steelblue"], edgecolor="black")
ax.set_title("Kickstarter Success Distribution")
ax.set_xticklabels(["Failed (0)", "Successful (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Success rate: {df[TARGET].mean():.1%}")

Success rate: 50.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (6400, 8), Test: (1600, 8)
Train class distribution:
success
1    0.5
0    0.5
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `category` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~50/50 by design.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["goal_per_day"] = X_train["goal_usd"] / (X_train["duration_days"] + 1)
X_test["goal_per_day"] = X_test["goal_usd"] / (X_test["duration_days"] + 1)

X_train["log_goal"] = np.log1p(X_train["goal_usd"])
X_test["log_goal"] = np.log1p(X_test["goal_usd"])

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (10): ['goal_usd', 'duration_days', 'category', 'backers_first_week', 'has_video', 'description_length', 'num_rewards', 'creator_prev_projects', 'goal_per_day', 'log_goal']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.6075
F1       : 0.6050

              precision    recall  f1-score   support

           0       0.61      0.61      0.61       800
           1       0.61      0.60      0.61       800

    accuracy                           0.61      1600
   macro avg       0.61      0.61      0.61      1600
weighted avg       0.61      0.61      0.61      1600



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                       
AdaBoostClassifier          0.611250           0.611250  0.646137  0.611235   0.611267  0.611250    0.276690
NearestCentroid             0.610000           0.610000  0.647006  0.609999   0.610001  0.610000    0.031981
LinearDiscriminantAnalysis  0.608750           0.608750  0.651952  0.608720   0.608783  0.608750    0.026448
CalibratedClassifierCV      0.608750           0.608750  0.651933  0.608720   0.608783  0.608750    0.056857
RidgeClassifier             0.608750           0.608750  0.651959  0.608720   0.608783  0.608750    0.018019
RidgeClassifierCV           0.608750           0.608750  0.651980  0.608720   0.608783  0.608750    0.019391
LogisticRegression          0.608125           0.608125  0.652077  0.608099   0.608154  0.6081

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.6006
F1: 0.5984


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.5978  (1.3s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[46]	valid_0's binary_logloss: 0.664161
LightGBM F1: 0.5913  (0.8s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.6075  0.6050     0.6089  0.6012
FLAML                  0.6006  0.5984     0.6018  0.5950
CatBoost               0.6031  0.5978     0.6059  0.5900
LightGBM               0.5956  0.5913     0.5977  0.5850

Top 3 models: ['Logistic Regression', 'FLAML', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  Logistic Regression:
    Accuracy : 0.6075
    F1       : 0.6050
    Precision: 0.6089
    Recall   : 0.6012

  FLAML:
    Accuracy : 0.6006
    F1       : 0.5984
    Precision: 0.6018
    Recall   : 0.5950

  CatBoost:
    Accuracy : 0.6031
    F1       : 0.5978
    Precision: 0.6059
    Recall   : 0.5900


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.61      0.61       800
           1       0.61      0.60      0.61       800

    accuracy                           0.61      1600
   macro avg       0.61      0.61      0.61      1600
weighted avg       0.61      0.61      0.61      1600


Total errors: 628 / 1600 (39.2%)

Sample misclassifications:
      goal_usd  duration_days  category  backers_first_week  has_video  description_length  num_rewards  creator_prev_projects  goal_per_day  log_goal  actual  predicted  correct
6860    1107.0             58       3.0                  10          1                 327           11                      1     18.762712  7.010312       1          0    False
6778    8014.0             23       3.0                  17          1                 320           10                      1    333.916667  8.989070       0          1    False
4374    9831.0    

## 25 · Interpretation and Business Insight

**Key findings:**
- **Lower goals** have significantly higher success rates.
- **Early backer momentum** (first week) is the strongest predictor.
- **Having a video** substantially improves chances.
- **Creator experience** (previous projects) adds confidence.
- **Games and Technology** have higher success rates.

**Business takeaway:** Set realistic goals, invest in video, build early momentum, and leverage creator track record.

## 26 · Limitations

1. Synthetic data with simplified success factors.
2. No pledge amount or backer contribution data.
3. Missing social media and marketing reach.
4. No text analysis of project descriptions.
5. Category effects are simplified.

## 27 · How to Improve This Project

1. Use real Kickstarter dataset from Web Robots.
2. Add social media follower counts.
3. Include text features from descriptions.
4. Add temporal features (launch day, time of year).
5. Model funding trajectory, not just binary outcome.

## 28 · Production Considerations

- Deploy as a campaign optimization tool.
- Output success probability with recommendations.
- Track real-time backer momentum.
- Update model quarterly with new campaigns.
- Segment by category for more accurate predictions.

## 29 · Common Mistakes

1. Using post-launch data (backers_first_week is borderline).
2. Not log-transforming highly skewed goal amounts.
3. Treating all categories equally.
4. Ignoring the temporal nature of crowdfunding.
5. Not considering campaign quality (photos, updates).

## 30 · Mini Challenge / Exercises

1. Remove `backers_first_week` — how much does F1 drop?
2. Log-transform `goal_usd` and compare results.
3. Separate analysis for Technology vs. Art categories.
4. Try different goal thresholds ($1K, $10K, $100K) as filters.
5. Plot success rate vs. goal amount in bins.

## 31 · Final Summary / Key Takeaways

1. **Goal amount** (lower is better) strongly predicts success.
2. **Early backer momentum** is the strongest signal.
3. **Video and description quality** matter.
4. **Creator experience** builds credibility.
5. **Category** affects success rates significantly.
6. **Campaign optimization** based on model insights can improve outcomes.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Kickstarter Success Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.6031,
    "f1": 0.5978,
    "precision": 0.6059,
    "recall": 0.59
  },
  "LightGBM": {
    "accuracy": 0.5956,
    "f1": 0.5913,
    "precision": 0.5977,
    "recall": 0.585
  },
  "Logistic Regression": {
    "accuracy": 0.6075,
    "f1": 0.605,
    "precision": 0.6089,
    "recall": 0.6012
  },
  "FLAML": {
    "accuracy": 0.6006,
    "f1": 0.5984,
    "precision": 0.6018,
    "recall": 0.595
  }
}
